# 02 — ROI Detection: YOLO Single-Class

Detect the "Reading Digit" window (ROI) using YOLOv11 trained on a single class.
Two experiments: utility-meter dataset (sparse ROI) and waterMeterDataset (full polygon ROI).

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")

    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(
            ["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"],
            check=True
        )

    BRANCH = "feature/roi-detection"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ultralytics", "albumentations", "rapidfuzz", "shapely"],
        check=True
    )

    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..").resolve()
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import yaml
import json
import time
import csv
import torch
import cv2
import numpy as np
from datetime import datetime
from ultralytics import YOLO

from models.data.unified_loader import load_water_meter_dataset_split
from models.data.roi_dataset import (
    filter_utility_meter_roi_samples,
    prepare_yolo_roi_dataset,
    prepare_wm_yolo_roi_dataset,
    polygon_to_bbox,
)
from models.metrics.evaluation import compute_iou_bbox

print(f"ROOT: {ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ROOT: C:\Users\alike\WaterMeterCV
CUDA available: True
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ===================================================================
# MODEL SIZE - change to switch:
#   "yolo11n" - nano  (~2.6M params)  <- start here
#   "yolo11s" - small (~9.6M params)  <- better accuracy
# ===================================================================
MODEL_SIZE = "yolo11n"

UM_YOLO_PATH = DATA_ROOT / "utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11"
WM_PATH = DATA_ROOT / "waterMeterDataset/WaterMeters"

WEIGHTS_DIR = WEIGHTS_BASE / "roi_yolo"
UM_ROI_DATASET = DATA_ROOT / "_roi_only_yolo_um"
WM_ROI_DATASET = DATA_ROOT / "_roi_only_yolo_wm"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATIENCE = 10

RUN_NAME = f"yolo_roi_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Model: {MODEL_SIZE}")
print(f"Device: {DEVICE}")
print(f"Workers: {WORKERS}")

Model: yolo11n
Device: cuda
Workers: 0


In [3]:
# UM: filter to ROI-only single-class YOLO dataset
prepare_yolo_roi_dataset(UM_YOLO_PATH, UM_ROI_DATASET)
um_train_roi = filter_utility_meter_roi_samples(UM_YOLO_PATH, "train")
um_test_roi  = filter_utility_meter_roi_samples(UM_YOLO_PATH, "test")
print(f"UM ROI: {len(um_train_roi)} train, {len(um_test_roi)} test")

# WM: 70/30 split, convert polygons to YOLO labels
wm_train, wm_test = load_water_meter_dataset_split(WM_PATH, train_ratio=0.7, seed=42)
prepare_wm_yolo_roi_dataset(wm_train, wm_test, WM_ROI_DATASET)
print(f"WM ROI: {len(wm_train)} train, {len(wm_test)} test")

UM ROI: 45 train, 6 test
WM ROI: 870 train, 374 test


## Experiment 1: utility-meter dataset

In [4]:
for split in ["train", "valid", "test"]:
    img_dir = UM_ROI_DATASET / split / "images"
    if img_dir.exists():
        count = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        print(f"  {split}: {count} images with ROI")

  train: 45 images with ROI
  valid: 6 images with ROI
  test: 6 images with ROI


## Training (utility-meter)

In [5]:
um_model = YOLO(f"{MODEL_SIZE}.pt")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

um_results = um_model.train(
    data=str(UM_ROI_DATASET / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(WEIGHTS_DIR),
    name=f"um_{RUN_NAME}",
    device=DEVICE,
    patience=PATIENCE,
    workers=WORKERS,
    save=True,
)

um_best = WEIGHTS_DIR / f"um_{RUN_NAME}" / "weights" / "best.pt"
print(f"Best weights: {um_best}")

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.13.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\alike\WaterMeterCV\WaterMetricsDATA\_roi_only_yolo_um\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum

## Evaluation (utility-meter)

In [6]:
um_best_model = YOLO(str(um_best))

um_ious = []
um_inference_times = []
for img_path, gt_bbox in um_test_roi:
    t0 = time.perf_counter()
    result = um_best_model.predict(str(img_path), verbose=False)[0]
    um_inference_times.append((time.perf_counter() - t0) * 1000)

    if result.boxes is not None and len(result.boxes) > 0:
        best_idx = result.boxes.conf.argmax()
        box = result.boxes.xywhn[best_idx]
        pred_bbox = (box[0].item(), box[1].item(), box[2].item(), box[3].item())
        um_ious.append(compute_iou_bbox(pred_bbox, gt_bbox))
    else:
        um_ious.append(0.0)

um_mean_iou = np.mean(um_ious) if um_ious else 0.0
um_det_rate = sum(1 for v in um_ious if v > 0) / len(um_ious) if um_ious else 0.0
um_avg_ms   = np.mean(um_inference_times) if um_inference_times else 0.0

print(f"UM — Mean IoU:        {um_mean_iou:.4f}")
print(f"UM — Detection rate:  {um_det_rate:.4f} ({sum(1 for v in um_ious if v > 0)}/{len(um_ious)})")
print(f"UM — Avg inference:   {um_avg_ms:.1f} ms/image")

UM — Mean IoU:        0.0000
UM — Detection rate:  0.0000 (0/6)
UM — Avg inference:   69.4 ms/image


## Experiment 2: waterMeterDataset

In [7]:
wm_model = YOLO(f"{MODEL_SIZE}.pt")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

wm_results = wm_model.train(
    data=str(WM_ROI_DATASET / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(WEIGHTS_DIR),
    name=f"wm_{RUN_NAME}",
    device=DEVICE,
    patience=PATIENCE,
    workers=WORKERS,
    save=True,
)

wm_best = WEIGHTS_DIR / f"wm_{RUN_NAME}" / "weights" / "best.pt"
print(f"Best weights: {wm_best}")

Ultralytics 8.4.33  Python-3.13.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\alike\WaterMeterCV\WaterMetricsDATA\_roi_only_yolo_wm\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=wm_yolo_roi_20260412_230832, nbs=64, nms=False, opset=None

In [8]:
wm_best_model = YOLO(str(wm_best))

wm_ious = []
wm_inference_times = []
for sample in wm_test:
    if sample.roi_polygon is None:
        continue
    gt_bbox = polygon_to_bbox(sample.roi_polygon)

    t0 = time.perf_counter()
    result = wm_best_model.predict(str(sample.image_path), verbose=False)[0]
    wm_inference_times.append((time.perf_counter() - t0) * 1000)

    if result.boxes is not None and len(result.boxes) > 0:
        best_idx = result.boxes.conf.argmax()
        box = result.boxes.xywhn[best_idx]
        pred_bbox = (box[0].item(), box[1].item(), box[2].item(), box[3].item())
        wm_ious.append(compute_iou_bbox(pred_bbox, gt_bbox))
    else:
        wm_ious.append(0.0)

wm_mean_iou = np.mean(wm_ious) if wm_ious else 0.0
wm_det_rate = sum(1 for v in wm_ious if v > 0) / len(wm_ious) if wm_ious else 0.0
wm_avg_ms   = np.mean(wm_inference_times) if wm_inference_times else 0.0

print(f"WM — Mean IoU:        {wm_mean_iou:.4f}")
print(f"WM — Detection rate:  {wm_det_rate:.4f} ({sum(1 for v in wm_ious if v > 0)}/{len(wm_ious)})")
print(f"WM — Avg inference:   {wm_avg_ms:.1f} ms/image")

WM — Mean IoU:        0.9400
WM — Detection rate:  1.0000 (374/374)
WM — Avg inference:   22.9 ms/image


## Predictions

In [9]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Top row: UM test samples
for i, ax in enumerate(axes[0]):
    if i >= len(um_test_roi):
        ax.axis("off")
        continue
    img_path, gt_bbox = um_test_roi[i]
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img = img.shape[:2]

    cx, cy, w, h = gt_bbox
    x1, y1 = int((cx - w/2) * w_img), int((cy - h/2) * h_img)
    x2, y2 = int((cx + w/2) * w_img), int((cy + h/2) * h_img)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    result = um_best_model.predict(str(img_path), verbose=False)[0]
    if result.boxes is not None and len(result.boxes) > 0:
        best_idx = result.boxes.conf.argmax()
        box = result.boxes.xywhn[best_idx]
        px, py, pw, ph = box[0].item(), box[1].item(), box[2].item(), box[3].item()
        px1, py1 = int((px - pw/2) * w_img), int((py - ph/2) * h_img)
        px2, py2 = int((px + pw/2) * w_img), int((py + ph/2) * h_img)
        cv2.rectangle(img, (px1, py1), (px2, py2), (255, 0, 0), 2)

    ax.imshow(img)
    iou_val = um_ious[i] if i < len(um_ious) else 0
    ax.set_title(f"UM IoU={iou_val:.2f}", fontsize=10)
    ax.axis("off")

# Bottom row: WM test samples
wm_test_with_roi = [s for s in wm_test if s.roi_polygon is not None]
for i, ax in enumerate(axes[1]):
    if i >= len(wm_test_with_roi):
        ax.axis("off")
        continue
    sample = wm_test_with_roi[i]
    img = cv2.imread(str(sample.image_path))
    if img is None:
        ax.axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img = img.shape[:2]

    gt_bbox = polygon_to_bbox(sample.roi_polygon)
    cx, cy, w, h = gt_bbox
    x1, y1 = int((cx - w/2) * w_img), int((cy - h/2) * h_img)
    x2, y2 = int((cx + w/2) * w_img), int((cy + h/2) * h_img)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    result = wm_best_model.predict(str(sample.image_path), verbose=False)[0]
    if result.boxes is not None and len(result.boxes) > 0:
        best_idx = result.boxes.conf.argmax()
        box = result.boxes.xywhn[best_idx]
        px, py, pw, ph = box[0].item(), box[1].item(), box[2].item(), box[3].item()
        px1, py1 = int((px - pw/2) * w_img), int((py - ph/2) * h_img)
        px2, py2 = int((px + pw/2) * w_img), int((py + ph/2) * h_img)
        cv2.rectangle(img, (px1, py1), (px2, py2), (255, 0, 0), 2)

    ax.imshow(img)
    iou_val = wm_ious[i] if i < len(wm_ious) else 0
    ax.set_title(f"WM IoU={iou_val:.2f}", fontsize=10)
    ax.axis("off")

plt.suptitle(f"YOLO ROI ({MODEL_SIZE}) — Green=GT, Red=Pred", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roi_yolo_predictions.png", dpi=150)
plt.close()
print("Saved to results/roi_yolo_predictions.png")

Saved to results/roi_yolo_predictions.png


## Comparison

In [10]:
print(f"{'='*60}")
print(f"{'Metric':<25} {'utility-meter':>15} {'waterMeter':>15}")
print(f"{'='*60}")
print(f"{'Mean IoU':<25} {um_mean_iou:>15.4f} {wm_mean_iou:>15.4f}")
print(f"{'Detection rate':<25} {um_det_rate:>15.4f} {wm_det_rate:>15.4f}")
print(f"{'Avg inference (ms)':<25} {um_avg_ms:>15.1f} {wm_avg_ms:>15.1f}")
print(f"{'N test':<25} {len(um_test_roi):>15d} {len(wm_test):>15d}")
print(f"{'='*60}")

Metric                      utility-meter      waterMeter
Mean IoU                           0.0000          0.9400
Detection rate                     0.0000          1.0000
Avg inference (ms)                   69.4            22.9
N test                                  6             374


## Save Results

In [11]:
metrics = {
    "method": "yolo_roi",
    "model_size": MODEL_SIZE,
    "utility_meter": {
        "mean_iou": round(um_mean_iou, 4),
        "detection_rate": round(um_det_rate, 4),
        "avg_inference_ms": round(um_avg_ms, 1),
        "n_train": len(um_train_roi),
        "n_test": len(um_test_roi),
    },
    "water_meter": {
        "mean_iou": round(wm_mean_iou, 4),
        "detection_rate": round(wm_det_rate, 4),
        "avg_inference_ms": round(wm_avg_ms, 1),
        "n_train": len(wm_train),
        "n_test": len(wm_test),
    },
    "config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "patience": PATIENCE,
    },
    "run_date": datetime.now().isoformat(),
}

with open(RESULTS_DIR / "roi_yolo_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "roi_comparison.csv"
csv_exists = csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    if not csv_exists:
        writer.writerow([
            "method", "um_mean_iou", "um_detection_rate", "um_inference_ms",
            "wm_mean_iou", "wm_detection_rate", "wm_inference_ms", "run_date",
        ])
    writer.writerow([
        "yolo_roi",
        round(um_mean_iou, 4), round(um_det_rate, 4), round(um_avg_ms, 1),
        round(wm_mean_iou, 4), round(wm_det_rate, 4), round(wm_avg_ms, 1),
        datetime.now().strftime("%Y-%m-%d %H:%M"),
    ])

print(f"Metrics -> {RESULTS_DIR / 'roi_yolo_metrics.json'}")
print(f"CSV    -> {csv_path}")

Metrics -> C:\Users\alike\WaterMeterCV\results\roi_yolo_metrics.json
CSV    -> C:\Users\alike\WaterMeterCV\results\roi_comparison.csv


## Conclusions

- **YOLO ROI (utility-meter):** Mean IoU=0.0000, Detection rate=0.0000
- **YOLO ROI (waterMeter):** Mean IoU=0.9400, Detection rate=1.0000
- **Inference:** 69.4 ms/image (UM), 22.9 ms/image (WM)
- **Next step:** compare with Faster R-CNN and U-Net